# Tentativi random di variational bayes

In [9]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

In [7]:
class Reservoir(nn.Module):
    def __init__(self, N, K, spectral_radius=0.9):
        super(Reservoir, self).__init__()
        
        self.N = N  # Number of reservoir neurons
        self.K = K  # Input dimension

        
        # Initialize Input Matrix (Uniform distribution is common)
        W_in = torch.rand(N, K) * 2 - 1  # Range [-1, 1]
        
        # Initialize Reservoir Matrix (Gaussian)
        W = torch.randn(N, N)
        
        # Scale Spectral Radius for stability
        # We calculate the largest eigenvalue and scale W
        with torch.no_grad():
            # Use real/imaginary parts to find the magnitude of eigenvalues
            eigenvalues = torch.linalg.eigvals(W)
            max_eig = torch.max(torch.abs(eigenvalues))
            W = W * (spectral_radius / max_eig)
        
        # Register as buffers (Fixed weights, move with model to GPU)
        self.register_buffer('W_in', W_in)
        self.register_buffer('W', W)
        
        # Internal state (Initialized to zeros)
        self.register_buffer('states', torch.zeros(1, N))
        
        self.activation = torch.tanh

    def forward(self, x):
        """
        x shape: [Batch, K]
        Returns the updated state: [Batch, N]
        """
        
        # Linear combinations: Input effect + Reservoir recurrent effect
        # We use .t() on weights because x is [Batch, K] and states is [Batch, N]
        input_part = x @ self.W_in.t()
        recurrent_part = self.states @ self.W.t()
        
        # We .detach() to ensure we don't track gradients through time steps
        self.states = self.activation(input_part + recurrent_part).detach()
        
        return self.states

    def reset_state(self, batch_size=1):
        """Clears the memory of the reservoir."""
        self.states = torch.zeros(batch_size, self.N, device=self.W.device)

The output matrix of the states update has dimension (batch_size,N)

## Dataset class

Each sample is of the type **($S_t$,$y_{t+1}$)**

In [4]:
from torch.utils.data import Dataset

class ESN_dataset(Dataset):
    # constructor
    def __init__(self, states, predictions):
        #inherit all the base class methods
        super().__init__()
        # store states and predictions
        self.states = states
        self.predictions = predictions

    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, index):
        x = self.states[index]
        y = self.predictions[index]

        return x,y
    

## Create dataset for a 1D time series

In [29]:
# reservoir parameters
N = 50
K = 1

L = 100 # time series length

times = torch.linspace(0,10,L)
time_series = torch.sin(times) + torch.randn(L) * 0.1
time_series = time_series.view(L, 1, 1)


In [30]:
model = Reservoir(N = N, K = K)
model.reset_state(batch_size=1)

In [33]:
states = torch.zeros(L,N)
for t in range(L):
    sample = time_series[t]
    state = model(sample)
    states[t] = state

In [55]:
data = ESN_dataset(states,time_series[1:].view(-1))

## Variational inference with Pyro